# Indian Startup Funding — SQL Analysis

**Notebook 03 of 4** | Goal: Answer structured business questions using SQL on the cleaned dataset.

## Why SQL After Python?
Python was for cleaning and transformation — manipulating data into shape.
SQL is for analysis — asking structured questions and getting precise answers.

In a real DA role at a product company (Swiggy, Razorpay, Zomato), you'd use SQL daily to answer questions like:
- "Which cities drove the most funding last year?"
- "How has the Fintech sector grown quarter over quarter?"
- "Which investors were most active during the funding winter?"

This notebook demonstrates that skill — every query answers a real business question.

## Structure
Each section has:
1. The **business question** in plain English
2. The **SQL query** with comments explaining every clause
3. The **output** interpreted in plain English

## Queries We'll Write
1. Total funding and deal count by year
2. Sector share evolution over time
3. Top 10 cities by total funding
4. Stage distribution by year
5. Top 20 investors by deal count
6. Median vs mean deal size by sector
7. Year-over-year growth rate (window function)
8. Running cumulative funding by year (window function)
9. Sector ranking per year using RANK() (window function)
10. Top funded company per sector using ROW_NUMBER() (window function)
11. Funding gap analysis using LAG() (window function)
12. Deal size buckets using CASE
13. City concentration index
14. Investor activity across funding eras
15. Cohort analysis — companies by first funding year

In [1]:
# ─── Imports ──────────────────────────────────────────────
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Display settings ─────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ─── Connect to SQLite database ───────────────────────────
DB_PATH = '../data/processed/funding.db'
conn = sqlite3.connect(DB_PATH)

print(f"✓ Connected to: {DB_PATH}")

# Verify the table exists and check row count
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM funding_rounds", conn)
print(f"✓ funding_rounds table: {result['total_rows'][0]:,} rows")

# Preview the table structure
print("\nTable columns:")
schema = pd.read_sql("PRAGMA table_info(funding_rounds)", conn)
for _, row in schema.iterrows():
    print(f"  {row['name']:<25} {row['type']}")

✓ Connected to: ../data/processed/funding.db
✓ funding_rounds table: 3,998 rows

Table columns:
  startup_name              TEXT
  startup_name_clean        TEXT
  industry                  TEXT
  sector                    TEXT
  city                      TEXT
  investors                 TEXT
  stage                     TEXT
  stage_clean               TEXT
  amount_usd                REAL
  date                      TEXT
  year                      REAL
  quarter                   REAL
  month                     REAL
  funding_era               TEXT
  source                    TEXT
  city_clean                TEXT


In [2]:
# ─── Query helper function ────────────────────────────────

def run_query(sql, description=""):
    """
    Runs a SQL query and returns a pandas DataFrame.
    Prints the description and row count of result.
    """
    if description:
        print(f"Query: {description}")
        print("=" * 60)
    
    result = pd.read_sql(sql, conn)
    print(f"Returned {len(result)} rows\n")
    return result

print("Helper function ready.")

Helper function ready.


---

## Query 1 — Total Funding and Deal Count By Year

**Business question:** How has the total capital deployed and number of deals changed year over year?

**SQL concepts used:** `WHERE`, `GROUP BY`, `ORDER BY`, aggregate functions (`SUM`, `COUNT`, `AVG`)

In [3]:
q1 = run_query("""
    SELECT 
        CAST(year AS INTEGER)            AS year,
        COUNT(*)                         AS total_deals,
        COUNT(amount_usd)                AS deals_with_amount,
        ROUND(SUM(amount_usd)/1e6, 1)    AS total_funding_mn_usd,
        ROUND(AVG(amount_usd)/1e6, 2)    AS avg_deal_size_mn_usd,
        ROUND(
            (COUNT(amount_usd) * 100.0) 
            / COUNT(*), 1
        )                                AS pct_disclosed
    FROM funding_rounds
    WHERE year >= 2010
      AND year <= 2020
    GROUP BY year
    ORDER BY year
""", "Total funding and deal count by year")

print(q1.to_string(index=False))

Query: Total funding and deal count by year
Returned 11 rows

 year  total_deals  deals_with_amount  total_funding_mn_usd  avg_deal_size_mn_usd  pct_disclosed
 2010           70                 70              1,007.80                 14.40         100.00
 2011          125                125              3,092.00                 24.74         100.00
 2012          135                135              1,130.80                  8.38         100.00
 2013          201                201              2,605.00                 12.96         100.00
 2014          191                191              1,691.40                  8.86         100.00
 2015          929                642              8,597.20                 13.39          69.10
 2016          993                586              3,828.10                  6.53          59.00
 2017          687                456             10,429.30                 22.87          66.40
 2018          309                264              5,116.10      

In [4]:
# ─── Save query to sql/ folder ────────────────────────────

query_01 = """-- Query 01: Total Funding and Deal Count By Year
-- Business question: How has capital deployment changed year over year?
-- Concepts: WHERE, GROUP BY, ORDER BY, SUM, COUNT, AVG, ROUND, CAST

SELECT 
    CAST(year AS INTEGER)            AS year,
    COUNT(*)                         AS total_deals,
    COUNT(amount_usd)                AS deals_with_amount,
    ROUND(SUM(amount_usd)/1e6, 1)    AS total_funding_mn_usd,
    ROUND(AVG(amount_usd)/1e6, 2)    AS avg_deal_size_mn_usd,
    ROUND(
        (COUNT(amount_usd) * 100.0) 
        / COUNT(*), 1
    )                                AS pct_disclosed
FROM funding_rounds
WHERE year >= 2010
  AND year <= 2020
GROUP BY year
ORDER BY year;
"""

with open('../sql/01_yearly_trends.sql', 'w') as f:
    f.write(query_01)

print("✓ Saved: sql/01_yearly_trends.sql")

✓ Saved: sql/01_yearly_trends.sql


---

## Query 2 — Sector Share Evolution Over Time

**Business question:** Which sectors grew, which shrank, which emerged across the decade?

**SQL concepts used:** `GROUP BY` multiple columns, `ROUND`, calculating percentages within groups using subquery

In [5]:
q2 = run_query("""
    SELECT
        CAST(year AS INTEGER)          AS year,
        sector,
        COUNT(*)                       AS deals,
        ROUND(SUM(amount_usd)/1e6, 1)  AS funding_mn_usd,
        ROUND(
            COUNT(*) * 100.0 / 
            SUM(COUNT(*)) OVER (PARTITION BY year)
        , 1)                           AS pct_of_year_deals
    FROM funding_rounds
    WHERE year >= 2014
      AND year <= 2020
      AND sector IS NOT NULL
    GROUP BY year, sector
    ORDER BY year, deals DESC
""", "Sector share evolution by year")

print(q2.to_string(index=False))

Query: Sector share evolution by year
Returned 77 rows

 year        sector  deals  funding_mn_usd  pct_of_year_deals
 2014         Other     49          656.00              27.40
 2014      SaaS/B2B     24          170.00              13.40
 2014    E-commerce     21          160.20              11.70
 2014    Healthtech     20          138.80              11.20
 2014 Media/Content     19          127.80              10.60
 2014        Edtech     13           17.10               7.30
 2014    Technology      9            5.50               5.00
 2014 Consumer Tech      7           60.70               3.90
 2014       Fintech      5           70.90               2.80
 2014      Proptech      5           67.30               2.80
 2014      Foodtech      3           17.90               1.70
 2014     Logistics      2           18.00               1.10
 2014      Mobility      2           36.70               1.10
 2015    Technology    171          630.50              22.50
 2015    E-com

In [6]:
query_02 = """-- Query 02: Sector Share Evolution Over Time
-- Business question: Which sectors grew, shrank, or emerged across the decade?
-- Concepts: GROUP BY multiple columns, window function for percentage calculation

SELECT
    CAST(year AS INTEGER)          AS year,
    sector,
    COUNT(*)                       AS deals,
    ROUND(SUM(amount_usd)/1e6, 1)  AS funding_mn_usd,
    ROUND(
        COUNT(*) * 100.0 / 
        SUM(COUNT(*)) OVER (PARTITION BY year)
    , 1)                           AS pct_of_year_deals
FROM funding_rounds
WHERE year >= 2014
  AND year <= 2020
  AND sector IS NOT NULL
GROUP BY year, sector
ORDER BY year, deals DESC;
"""

with open('../sql/02_sector_evolution.sql', 'w') as f:
    f.write(query_02)

print("✓ Saved: sql/02_sector_evolution.sql")

✓ Saved: sql/02_sector_evolution.sql


---

## Query 3 — Top 10 Cities by Total Funding

**Business question:** Which cities dominate Indian startup funding — and by how much?

**SQL concepts used:** `GROUP BY`, `ORDER BY DESC`, `LIMIT`, `HAVING` to filter groups

In [7]:
q3 = run_query("""
    SELECT
        city_clean                        AS city,
        COUNT(*)                          AS total_deals,
        ROUND(SUM(amount_usd)/1e6, 1)     AS total_funding_mn_usd,
        ROUND(AVG(amount_usd)/1e6, 2)     AS avg_deal_size_mn_usd,
        ROUND(
            SUM(amount_usd) * 100.0 /
            SUM(SUM(amount_usd)) OVER ()
        , 1)                              AS pct_of_total_funding
    FROM funding_rounds
    WHERE city_clean IS NOT NULL
      AND amount_usd IS NOT NULL
    GROUP BY city_clean
    HAVING COUNT(*) >= 10
    ORDER BY total_funding_mn_usd DESC
    LIMIT 10
""", "Top 10 cities by total funding")

print(q3.to_string(index=False))

Query: Top 10 cities by total funding
Returned 10 rows

     city  total_deals  total_funding_mn_usd  avg_deal_size_mn_usd  pct_of_total_funding
Bengaluru          847             24,357.00                 28.76                 49.40
   Mumbai          600              6,930.10                 11.55                 14.10
    Delhi          373              6,708.40                 17.99                 13.60
  Gurgaon          327              5,453.00                 16.68                 11.10
  Chennai          146              1,732.30                 11.87                  3.50
    Noida           90              1,620.90                 18.01                  3.30
Hyderabad          132              1,155.70                  8.76                  2.30
     Pune           93                876.40                  9.42                  1.80
   Jaipur           20                222.10                 11.11                  0.50
Ahmedabad           37                128.00          

In [8]:
query_03 = """-- Query 03: Top 10 Cities by Total Funding
-- Business question: Which cities dominate Indian startup funding?
-- Concepts: GROUP BY, HAVING, LIMIT, window function for grand total percentage

SELECT
    city_clean                        AS city,
    COUNT(*)                          AS total_deals,
    ROUND(SUM(amount_usd)/1e6, 1)     AS total_funding_mn_usd,
    ROUND(AVG(amount_usd)/1e6, 2)     AS avg_deal_size_mn_usd,
    ROUND(
        SUM(amount_usd) * 100.0 /
        SUM(SUM(amount_usd)) OVER ()
    , 1)                              AS pct_of_total_funding
FROM funding_rounds
WHERE city_clean IS NOT NULL
  AND amount_usd IS NOT NULL
GROUP BY city_clean
HAVING COUNT(*) >= 10
ORDER BY total_funding_mn_usd DESC
LIMIT 10;
"""

with open('../sql/03_top_cities.sql', 'w') as f:
    f.write(query_03)

print("✓ Saved: sql/03_top_cities.sql")

✓ Saved: sql/03_top_cities.sql


---

## Query 4 — Year-over-Year Growth Rate

**Business question:** How did funding grow or shrink each year compared to the previous year?

**SQL concepts used:** CTE (`WITH`), `LAG()` window function, calculated growth rate

### What is LAG()?
`LAG(column, 1)` looks at the **previous row's value** in a defined order.
So if this year's funding is $8.6B and last year's was $1.7B,
`LAG` gives us $1.7B — and we can calculate growth rate from there.

This is impossible with a simple GROUP BY — you need a window function.

In [9]:
q4 = run_query("""
    WITH yearly_totals AS (
        SELECT
            CAST(year AS INTEGER)         AS year,
            COUNT(*)                      AS total_deals,
            ROUND(SUM(amount_usd)/1e6, 1) AS total_funding_mn_usd
        FROM funding_rounds
        WHERE year >= 2012
          AND year <= 2020
          AND amount_usd IS NOT NULL
        GROUP BY year
    )
    SELECT
        year,
        total_deals,
        total_funding_mn_usd,
        LAG(total_funding_mn_usd, 1) OVER (ORDER BY year) 
                                          AS prev_year_funding,
        ROUND(
            (total_funding_mn_usd - LAG(total_funding_mn_usd, 1) 
                OVER (ORDER BY year)) * 100.0 /
            LAG(total_funding_mn_usd, 1) OVER (ORDER BY year)
        , 1)                              AS yoy_growth_pct
    FROM yearly_totals
    ORDER BY year
""", "Year-over-year funding growth rate")

print(q4.to_string(index=False))

Query: Year-over-year funding growth rate
Returned 9 rows

 year  total_deals  total_funding_mn_usd  prev_year_funding  yoy_growth_pct
 2012          135              1,130.80                NaN             NaN
 2013          201              2,605.00           1,130.80          130.40
 2014          191              1,691.40           2,605.00          -35.10
 2015          642              8,597.20           1,691.40          408.30
 2016          586              3,828.10           8,597.20          -55.50
 2017          456             10,429.30           3,828.10          172.40
 2018          264              5,116.10          10,429.30          -50.90
 2019          104              9,686.60           5,116.10           89.30
 2020            7                390.20           9,686.60          -96.00


In [10]:
query_04 = """-- Query 04: Year-over-Year Funding Growth Rate
-- Business question: How did funding grow or shrink each year vs previous year?
-- Concepts: CTE (WITH clause), LAG() window function, growth rate calculation

WITH yearly_totals AS (
    SELECT
        CAST(year AS INTEGER)         AS year,
        COUNT(*)                      AS total_deals,
        ROUND(SUM(amount_usd)/1e6, 1) AS total_funding_mn_usd
    FROM funding_rounds
    WHERE year >= 2012
      AND year <= 2020
      AND amount_usd IS NOT NULL
    GROUP BY year
)
SELECT
    year,
    total_deals,
    total_funding_mn_usd,
    LAG(total_funding_mn_usd, 1) OVER (ORDER BY year)
                                      AS prev_year_funding,
    ROUND(
        (total_funding_mn_usd - LAG(total_funding_mn_usd, 1)
            OVER (ORDER BY year)) * 100.0 /
        LAG(total_funding_mn_usd, 1) OVER (ORDER BY year)
    , 1)                              AS yoy_growth_pct
FROM yearly_totals
ORDER BY year;
"""

with open('../sql/04_yoy_growth.sql', 'w') as f:
    f.write(query_04)

print("✓ Saved: sql/04_yoy_growth.sql")

✓ Saved: sql/04_yoy_growth.sql


---

## Query 5 — Sector Ranking Per Year

**Business question:** Which sector ranked #1 by deal count each year — and how did rankings shift?

**SQL concepts used:** CTE, `RANK()` window function with `PARTITION BY`

### What is RANK()?
`RANK() OVER (PARTITION BY year ORDER BY deals DESC)`

Means: "Within each year, rank sectors by deal count — highest = rank 1."

`PARTITION BY year` resets the ranking for every year.
Without it, you'd get one global ranking across all years combined.


In [12]:
q5 = run_query("""
    WITH sector_year AS (
        SELECT
            CAST(year AS INTEGER)         AS year,
            sector,
            COUNT(*)                      AS deals,
            ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
        FROM funding_rounds
        WHERE year >= 2014
          AND year <= 2020
          AND sector IS NOT NULL
        GROUP BY year, sector
    ),
    ranked AS (
        SELECT
            year,
            sector,
            deals,
            funding_mn_usd,
            RANK() OVER (
                PARTITION BY year
                ORDER BY deals DESC
            )                             AS rank_by_deals,
            RANK() OVER (
                PARTITION BY year
                ORDER BY funding_mn_usd DESC
            )                             AS rank_by_funding
        FROM sector_year
    )
    SELECT *
    FROM ranked
    WHERE rank_by_deals <= 5
    ORDER BY year, rank_by_deals
""", "Sector ranking per year")

print(q5.to_string(index=False))

Query: Sector ranking per year
Returned 37 rows

 year        sector  deals  funding_mn_usd  rank_by_deals  rank_by_funding
 2014         Other     49          656.00              1                1
 2014      SaaS/B2B     24          170.00              2                2
 2014    E-commerce     21          160.20              3                3
 2014    Healthtech     20          138.80              4                4
 2014 Media/Content     19          127.80              5                5
 2015    Technology    171          630.50              1                4
 2015    E-commerce    135        3,315.10              2                1
 2015         Other    131        1,112.40              3                2
 2015      Foodtech     64          605.40              4                5
 2015    Healthtech     49          204.40              5                7
 2016 Consumer Tech    539        1,918.80              1                1
 2016    Technology    190          682.50         

In [13]:
query_05 = """-- Query 05: Sector Ranking Per Year Using RANK()
-- Business question: Which sector ranked #1 each year by deals and funding?
-- Concepts: Multiple CTEs, RANK() with PARTITION BY, filtering on window function alias

WITH sector_year AS (
    SELECT
        CAST(year AS INTEGER)         AS year,
        sector,
        COUNT(*)                      AS deals,
        ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
    FROM funding_rounds
    WHERE year >= 2014
      AND year <= 2020
      AND sector IS NOT NULL
    GROUP BY year, sector
),
ranked AS (
    SELECT
        year,
        sector,
        deals,
        funding_mn_usd,
        RANK() OVER (
            PARTITION BY year
            ORDER BY deals DESC
        )                             AS rank_by_deals,
        RANK() OVER (
            PARTITION BY year
            ORDER BY funding_mn_usd DESC
        )                             AS rank_by_funding
    FROM sector_year
)
SELECT *
FROM ranked
WHERE rank_by_deals <= 5
ORDER BY year, rank_by_deals;
"""

with open('../sql/05_sector_rankings.sql', 'w') as f:
    f.write(query_05)

print("✓ Saved: sql/05_sector_rankings.sql")

✓ Saved: sql/05_sector_rankings.sql


---

## Query 6 — Top Funded Company Per Sector

**Business question:** Which company raised the most money in each sector?

**SQL concepts used:** CTE, `ROW_NUMBER()` window function

### RANK() vs ROW_NUMBER() — The Key Difference
Both assign numbers to rows in order. The difference is with ties:

| Score | RANK() | ROW_NUMBER() |
|---|---|---|
| 100 | 1 | 1 |
| 100 | 1 | 2 |
| 90  | 3 | 3 |

`RANK()` gives tied rows the same number and skips the next.
`ROW_NUMBER()` always gives unique numbers — ties broken arbitrarily.

**Use ROW_NUMBER() when you want exactly one row per group.**
**Use RANK() when you want all tied rows included.**

In [14]:
q6 = run_query("""
    WITH company_sector_total AS (
        SELECT
            startup_name,
            sector,
            COUNT(*)                       AS total_rounds,
            ROUND(SUM(amount_usd)/1e6, 1)  AS total_raised_mn_usd
        FROM funding_rounds
        WHERE sector IS NOT NULL
          AND amount_usd IS NOT NULL
        GROUP BY startup_name, sector
    ),
    ranked AS (
        SELECT
            startup_name,
            sector,
            total_rounds,
            total_raised_mn_usd,
            ROW_NUMBER() OVER (
                PARTITION BY sector
                ORDER BY total_raised_mn_usd DESC
            )                              AS rn
        FROM company_sector_total
    )
    SELECT
        sector,
        startup_name,
        total_rounds,
        total_raised_mn_usd
    FROM ranked
    WHERE rn = 1
    ORDER BY total_raised_mn_usd DESC
""", "Top funded company per sector")

print(q6.to_string(index=False))

Query: Top funded company per sector
Returned 14 rows

       sector             startup_name  total_rounds  total_raised_mn_usd
   E-commerce                 Flipkart            13             7,161.80
     Mobility         Rapido Bike Taxi             1             3,900.00
      Fintech                    Paytm             1             1,000.00
     SaaS/B2B                    Udaan             2               810.00
        Other               True North             1               600.00
Consumer Tech                      Ola             3               484.50
   Healthtech                    GOQii             1               450.00
     Foodtech                   Zomato             3               285.00
   Technology              ReNew Power             1               275.00
       Edtech Furtados School of Music             1               200.00
    Logistics             Ecom Express             3               183.50
Media/Content              Komli Media             6     

In [15]:
query_06 = """-- Query 06: Top Funded Company Per Sector
-- Business question: Which company raised the most in each sector?
-- Concepts: ROW_NUMBER() vs RANK(), PARTITION BY sector

WITH company_sector_total AS (
    SELECT
        startup_name,
        sector,
        COUNT(*)                       AS total_rounds,
        ROUND(SUM(amount_usd)/1e6, 1)  AS total_raised_mn_usd
    FROM funding_rounds
    WHERE sector IS NOT NULL
      AND amount_usd IS NOT NULL
    GROUP BY startup_name, sector
),
ranked AS (
    SELECT
        startup_name,
        sector,
        total_rounds,
        total_raised_mn_usd,
        ROW_NUMBER() OVER (
            PARTITION BY sector
            ORDER BY total_raised_mn_usd DESC
        )                              AS rn
    FROM company_sector_total
)
SELECT
    sector,
    startup_name,
    total_rounds,
    total_raised_mn_usd
FROM ranked
WHERE rn = 1
ORDER BY total_raised_mn_usd DESC;
"""

with open('../sql/06_top_company_per_sector.sql', 'w') as f:
    f.write(query_06)

print("✓ Saved: sql/06_top_company_per_sector.sql")

✓ Saved: sql/06_top_company_per_sector.sql


---

## Query 7 — Deal Size Distribution Using CASE

**Business question:** How are deals distributed across size buckets — micro, small, mid, large, mega?

**SQL concepts used:** `CASE` statement for conditional bucketing, `GROUP BY` on derived column

### What is CASE?
CASE is SQL's if/elif/else. It creates a new column based on conditions:

```sql
CASE
    WHEN amount < 1000000  THEN 'Micro'
    WHEN amount < 10000000 THEN 'Small'
    ELSE 'Large'
END
```


In [16]:
q7 = run_query("""
    WITH bucketed AS (
        SELECT
            startup_name,
            sector,
            amount_usd,
            CAST(year AS INTEGER) AS year,
            CASE
                WHEN amount_usd < 500000        THEN '1. Micro (<$500K)'
                WHEN amount_usd < 2000000       THEN '2. Small ($500K-$2M)'
                WHEN amount_usd < 10000000      THEN '3. Mid ($2M-$10M)'
                WHEN amount_usd < 50000000      THEN '4. Large ($10M-$50M)'
                WHEN amount_usd < 200000000     THEN '5. Growth ($50M-$200M)'
                ELSE                                 '6. Mega ($200M+)'
            END                                AS deal_size_bucket
        FROM funding_rounds
        WHERE amount_usd IS NOT NULL
          AND year >= 2014
          AND year <= 2020
    )
    SELECT
        deal_size_bucket,
        COUNT(*)                          AS deal_count,
        ROUND(COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (), 1)     AS pct_of_deals,
        ROUND(SUM(amount_usd)/1e6, 1)     AS total_funding_mn_usd,
        ROUND(SUM(amount_usd) * 100.0 /
            SUM(SUM(amount_usd)) OVER (), 1) AS pct_of_funding
    FROM bucketed
    GROUP BY deal_size_bucket
    ORDER BY deal_size_bucket
""", "Deal size distribution by bucket")

print(q7.to_string(index=False))

Query: Deal size distribution by bucket
Returned 6 rows

      deal_size_bucket  deal_count  pct_of_deals  total_funding_mn_usd  pct_of_funding
     1. Micro (<$500K)         567         25.20                120.60            0.30
  2. Small ($500K-$2M)         572         25.40                536.50            1.30
     3. Mid ($2M-$10M)         584         26.00              2,572.70            6.50
  4. Large ($10M-$50M)         378         16.80              7,433.90           18.70
5. Growth ($50M-$200M)         115          5.10              9,490.20           23.90
      6. Mega ($200M+)          34          1.50             19,585.00           49.30


In [17]:
query_07 = """-- Query 07: Deal Size Distribution Using CASE
-- Business question: How are deals distributed across size buckets?
-- Concepts: CASE statement for bucketing, window function for percentages

WITH bucketed AS (
    SELECT
        startup_name,
        sector,
        amount_usd,
        CAST(year AS INTEGER) AS year,
        CASE
            WHEN amount_usd < 500000        THEN '1. Micro (<$500K)'
            WHEN amount_usd < 2000000       THEN '2. Small ($500K-$2M)'
            WHEN amount_usd < 10000000      THEN '3. Mid ($2M-$10M)'
            WHEN amount_usd < 50000000      THEN '4. Large ($10M-$50M)'
            WHEN amount_usd < 200000000     THEN '5. Growth ($50M-$200M)'
            ELSE                                 '6. Mega ($200M+)'
        END                                AS deal_size_bucket
    FROM funding_rounds
    WHERE amount_usd IS NOT NULL
      AND year >= 2014
      AND year <= 2020
)
SELECT
    deal_size_bucket,
    COUNT(*)                          AS deal_count,
    ROUND(COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (), 1)     AS pct_of_deals,
    ROUND(SUM(amount_usd)/1e6, 1)     AS total_funding_mn_usd,
    ROUND(SUM(amount_usd) * 100.0 /
        SUM(SUM(amount_usd)) OVER (), 1) AS pct_of_funding
FROM bucketed
GROUP BY deal_size_bucket
ORDER BY deal_size_bucket;
"""

with open('../sql/07_deal_size_buckets.sql', 'w') as f:
    f.write(query_07)

print("✓ Saved: sql/07_deal_size_buckets.sql")

✓ Saved: sql/07_deal_size_buckets.sql


---

## Query 8 — Running Cumulative Funding By Year

**Business question:** How did total cumulative capital deployed grow across the decade?

**SQL concepts used:** `SUM() OVER (ORDER BY year)` — cumulative window function

### What is a Running Total?
Instead of funding PER year, this shows funding UP TO each year.
2014: $1.7B
2015: $1.7B + $8.6B = $10.3B total
2016: $10.3B + $3.8B = $14.1B total
...and so on.

This makes the growth curve visually dramatic in Tableau.

In [18]:
q8 = run_query("""
    WITH yearly AS (
        SELECT
            CAST(year AS INTEGER)         AS year,
            COUNT(*)                      AS deals,
            ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
        FROM funding_rounds
        WHERE year >= 2012
          AND year <= 2020
          AND amount_usd IS NOT NULL
        GROUP BY year
    )
    SELECT
        year,
        deals,
        funding_mn_usd,
        SUM(deals) OVER (
            ORDER BY year
        )                                 AS cumulative_deals,
        ROUND(SUM(funding_mn_usd) OVER (
            ORDER BY year
        ), 1)                             AS cumulative_funding_mn_usd
    FROM yearly
    ORDER BY year
""", "Running cumulative funding by year")

print(q8.to_string(index=False))

Query: Running cumulative funding by year
Returned 9 rows

 year  deals  funding_mn_usd  cumulative_deals  cumulative_funding_mn_usd
 2012    135        1,130.80               135                   1,130.80
 2013    201        2,605.00               336                   3,735.80
 2014    191        1,691.40               527                   5,427.20
 2015    642        8,597.20              1169                  14,024.40
 2016    586        3,828.10              1755                  17,852.50
 2017    456       10,429.30              2211                  28,281.80
 2018    264        5,116.10              2475                  33,397.90
 2019    104        9,686.60              2579                  43,084.50
 2020      7          390.20              2586                  43,474.70


In [19]:
query_08 = """-- Query 08: Running Cumulative Funding By Year
-- Business question: How did total capital deployed grow across the decade?
-- Concepts: SUM() OVER (ORDER BY) cumulative window function

WITH yearly AS (
    SELECT
        CAST(year AS INTEGER)         AS year,
        COUNT(*)                      AS deals,
        ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
    FROM funding_rounds
    WHERE year >= 2012
      AND year <= 2020
      AND amount_usd IS NOT NULL
    GROUP BY year
)
SELECT
    year,
    deals,
    funding_mn_usd,
    SUM(deals) OVER (
        ORDER BY year
    )                                 AS cumulative_deals,
    ROUND(SUM(funding_mn_usd) OVER (
        ORDER BY year
    ), 1)                             AS cumulative_funding_mn_usd
FROM yearly
ORDER BY year;
"""

with open('../sql/08_cumulative_funding.sql', 'w') as f:
    f.write(query_08)

print("✓ Saved: sql/08_cumulative_funding.sql")

✓ Saved: sql/08_cumulative_funding.sql


---

## Query 9 — Top Investors By Deal Count

**Business question:** Which investors were most active — and did they stay active across cycles?

**SQL concepts used:** `GROUP BY`, `HAVING`, `ORDER BY`, string filtering with `WHERE`

### Why This Matters For Product DA Roles
At Swiggy or Razorpay, understanding investor behavior tells you:
- Who has dry powder to deploy in the next cycle
- Which investors stayed disciplined vs chased hype
- Who exited during the winter vs doubled down

In [20]:
q9 = run_query("""
    SELECT
        investors,
        COUNT(*)                          AS total_deals,
        COUNT(DISTINCT startup_name)      AS unique_companies,
        ROUND(SUM(amount_usd)/1e6, 1)     AS total_invested_mn_usd,
        CAST(MIN(year) AS INTEGER)        AS first_deal_year,
        CAST(MAX(year) AS INTEGER)        AS last_deal_year,
        CAST(MAX(year) - MIN(year) AS INTEGER) AS years_active
    FROM funding_rounds
    WHERE investors IS NOT NULL
      AND investors != 'nan'
      AND amount_usd IS NOT NULL
      AND year >= 2014
      AND year <= 2020
    GROUP BY investors
    HAVING COUNT(*) >= 3
    ORDER BY total_deals DESC
    LIMIT 20
""", "Top 20 investors by deal count")

print(q9.to_string(index=False))

Query: Top 20 investors by deal count
Returned 20 rows

                investors  total_deals  unique_companies  total_invested_mn_usd  first_deal_year  last_deal_year  years_active
    Undisclosed Investors           29                28                  32.10             2015            2018             3
    Undisclosed investors           24                24                 291.10             2015            2017             2
          Sequoia Capital           13                13                 194.80             2015            2018             3
          Kalaari Capital           13                13                  24.60             2015            2016             1
 Group of Angel Investors           13                12                   1.50             2015            2015             0
    undisclosed investors           11                11                   8.40             2016            2017             1
     Indian Angel Network           11                1

In [21]:
q9_clean = run_query("""
    SELECT
        investors,
        COUNT(*)                          AS total_deals,
        COUNT(DISTINCT startup_name)      AS unique_companies,
        ROUND(SUM(amount_usd)/1e6, 1)     AS total_invested_mn_usd,
        CAST(MIN(year) AS INTEGER)        AS first_deal_year,
        CAST(MAX(year) AS INTEGER)        AS last_deal_year,
        CAST(MAX(year) - MIN(year) AS INTEGER) AS years_active
    FROM funding_rounds
    WHERE investors IS NOT NULL
      AND investors != 'nan'
      AND amount_usd IS NOT NULL
      AND year >= 2014
      AND year <= 2020
      AND LOWER(investors) NOT LIKE '%undisclosed%'
      AND LOWER(investors) NOT LIKE '%angel investor%'
      AND LOWER(investors) NOT LIKE '%group of%'
    GROUP BY investors
    HAVING COUNT(*) >= 3
    ORDER BY total_deals DESC
    LIMIT 20
""", "Top 20 real investors by deal count")

print(q9_clean.to_string(index=False))

Query: Top 20 real investors by deal count
Returned 20 rows

                                 investors  total_deals  unique_companies  total_invested_mn_usd  first_deal_year  last_deal_year  years_active
                           Sequoia Capital           13                13                 194.80             2015            2018             3
                           Kalaari Capital           13                13                  24.60             2015            2016             1
                      Indian Angel Network           11                11                   6.30             2015            2017             2
                             SAIF Partners           10                10                  18.50             2015            2019             4
                     Info Edge (India) Ltd            9                 7                  10.70             2015            2018             3
                            Accel Partners            9                 9  

In [22]:
query_09 = """-- Query 09: Top Investors By Deal Count
-- Business question: Which investors were most active across cycles?
-- Concepts: GROUP BY, HAVING, LOWER() for case-insensitive filtering, NOT LIKE

SELECT
    investors,
    COUNT(*)                          AS total_deals,
    COUNT(DISTINCT startup_name)      AS unique_companies,
    ROUND(SUM(amount_usd)/1e6, 1)     AS total_invested_mn_usd,
    CAST(MIN(year) AS INTEGER)        AS first_deal_year,
    CAST(MAX(year) AS INTEGER)        AS last_deal_year,
    CAST(MAX(year) - MIN(year) AS INTEGER) AS years_active
FROM funding_rounds
WHERE investors IS NOT NULL
  AND investors != 'nan'
  AND amount_usd IS NOT NULL
  AND year >= 2014
  AND year <= 2020
  AND LOWER(investors) NOT LIKE '%undisclosed%'
  AND LOWER(investors) NOT LIKE '%angel investor%'
  AND LOWER(investors) NOT LIKE '%group of%'
GROUP BY investors
HAVING COUNT(*) >= 3
ORDER BY total_deals DESC
LIMIT 20;
"""

with open('../sql/09_investor_activity.sql', 'w') as f:
    f.write(query_09)

print("✓ Saved: sql/09_investor_activity.sql")

✓ Saved: sql/09_investor_activity.sql


---

## Query 10 — Median vs Mean Deal Size By Sector

**Business question:** Which sectors have the most skewed deal sizes — where a few mega rounds distort the average?

**SQL concepts used:** Subquery for median calculation, `AVG()`, `COUNT()`, comparison of central tendency measures

### Why Median vs Mean Matters
Mean is pulled up by outliers. Median is not.

If Flipkart raises $1.5B and 9 other e-commerce startups raise $1M each:
- Mean = ($1.5B + $9M) / 10 = $150.9M — misleadingly high
- Median = $1M — the real typical deal

**When mean >> median, the sector has a few massive outliers dominating.**
This is the honest way to report funding data.

In [23]:
q10 = run_query("""
    WITH sector_stats AS (
        SELECT
            sector,
            amount_usd,
            COUNT(*) OVER (PARTITION BY sector)          AS sector_deal_count,
            ROW_NUMBER() OVER (
                PARTITION BY sector
                ORDER BY amount_usd
            )                                            AS rn,
            COUNT(*) OVER (PARTITION BY sector)          AS cnt
        FROM funding_rounds
        WHERE sector IS NOT NULL
          AND amount_usd IS NOT NULL
          AND year >= 2014
          AND year <= 2020
    ),
    medians AS (
        SELECT
            sector,
            AVG(amount_usd)                              AS median_usd
        FROM sector_stats
        WHERE rn IN (
            (cnt + 1) / 2,
            (cnt + 2) / 2
        )
        GROUP BY sector
    ),
    means AS (
        SELECT
            sector,
            COUNT(*)                                     AS deals,
            ROUND(AVG(amount_usd)/1e6, 2)                AS mean_mn_usd,
            ROUND(SUM(amount_usd)/1e6, 1)                AS total_mn_usd
        FROM funding_rounds
        WHERE sector IS NOT NULL
          AND amount_usd IS NOT NULL
          AND year >= 2014
          AND year <= 2020
        GROUP BY sector
    )
    SELECT
        m.sector,
        m.deals,
        ROUND(md.median_usd/1e6, 2)                      AS median_mn_usd,
        m.mean_mn_usd,
        ROUND(m.mean_mn_usd / (md.median_usd/1e6), 1)    AS mean_to_median_ratio,
        m.total_mn_usd
    FROM means m
    JOIN medians md ON m.sector = md.sector
    ORDER BY mean_to_median_ratio DESC
""", "Median vs mean deal size by sector")

print(q10.to_string(index=False))

Query: Median vs mean deal size by sector
Returned 14 rows

       sector  deals  median_mn_usd  mean_mn_usd  mean_to_median_ratio  total_mn_usd
     Mobility     31           6.50       180.90                 27.80      5,608.00
   E-commerce    311           3.00        37.68                 12.60     11,718.60
Consumer Tech    615           1.00        10.31                 10.30      6,340.10
      Fintech     44           4.50        35.84                  8.00      1,576.80
       Edtech     75           1.00         7.93                  7.90        594.50
     Foodtech     85           1.60        11.95                  7.50      1,015.70
     SaaS/B2B     53           4.00        28.71                  7.20      1,521.80
   Technology    449           1.10         7.59                  6.90      3,408.10
   Healthtech    108           2.25        13.93                  6.20      1,504.60
        Other    231           3.60        17.58                  4.90      4,061.20
Media

In [24]:
query_10 = """-- Query 10: Median vs Mean Deal Size By Sector
-- Business question: Which sectors have most skewed deal sizes?
-- Concepts: Median calculation using ROW_NUMBER(), JOIN between CTEs, ratio analysis

WITH sector_stats AS (
    SELECT
        sector,
        amount_usd,
        ROW_NUMBER() OVER (
            PARTITION BY sector
            ORDER BY amount_usd
        )                                            AS rn,
        COUNT(*) OVER (PARTITION BY sector)          AS cnt
    FROM funding_rounds
    WHERE sector IS NOT NULL
      AND amount_usd IS NOT NULL
      AND year >= 2014
      AND year <= 2020
),
medians AS (
    SELECT
        sector,
        AVG(amount_usd)                              AS median_usd
    FROM sector_stats
    WHERE rn IN (
        (cnt + 1) / 2,
        (cnt + 2) / 2
    )
    GROUP BY sector
),
means AS (
    SELECT
        sector,
        COUNT(*)                                     AS deals,
        ROUND(AVG(amount_usd)/1e6, 2)                AS mean_mn_usd,
        ROUND(SUM(amount_usd)/1e6, 1)                AS total_mn_usd
    FROM funding_rounds
    WHERE sector IS NOT NULL
      AND amount_usd IS NOT NULL
      AND year >= 2014
      AND year <= 2020
    GROUP BY sector
)
SELECT
    m.sector,
    m.deals,
    ROUND(md.median_usd/1e6, 2)                      AS median_mn_usd,
    m.mean_mn_usd,
    ROUND(m.mean_mn_usd / (md.median_usd/1e6), 1)    AS mean_to_median_ratio,
    m.total_mn_usd
FROM means m
JOIN medians md ON m.sector = md.sector
ORDER BY mean_to_median_ratio DESC;
"""

with open('../sql/10_median_vs_mean.sql', 'w') as f:
    f.write(query_10)

print("✓ Saved: sql/10_median_vs_mean.sql")

✓ Saved: sql/10_median_vs_mean.sql


---

## Query 11 — Funding Stage Evolution By Year

**Business question:** Did the Indian ecosystem mature over time — shifting from seed-heavy to growth-stage capital?

**SQL concepts used:** `GROUP BY` multiple columns, `ROUND`, percentage within year using window function

In [25]:
q11 = run_query("""
    WITH stage_year AS (
        SELECT
            CAST(year AS INTEGER)  AS year,
            stage_clean,
            COUNT(*)               AS deals,
            ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
        FROM funding_rounds
        WHERE year >= 2014
          AND year <= 2020
          AND stage_clean IS NOT NULL
          AND stage_clean != 'Other'
        GROUP BY year, stage_clean
    )
    SELECT
        year,
        stage_clean,
        deals,
        funding_mn_usd,
        ROUND(
            deals * 100.0 /
            SUM(deals) OVER (PARTITION BY year)
        , 1)                       AS pct_of_year_deals
    FROM stage_year
    ORDER BY year, deals DESC
""", "Funding stage evolution by year")

print(q11.to_string(index=False))

Query: Funding stage evolution by year
Returned 32 rows

 year  stage_clean  deals  funding_mn_usd  pct_of_year_deals
 2014     Series A     99        1,143.00              51.80
 2014         Seed     76           49.20              39.80
 2014     Series B      8          109.00               4.20
 2014      PE/Debt      4          280.80               2.10
 2014   Late Stage      2           67.00               1.00
 2014     Series C      2           42.30               1.00
 2015         Seed    494          161.40              53.20
 2015      PE/Debt    434        8,434.80              46.80
 2016         Seed    586          116.70              59.10
 2016      PE/Debt    406        3,711.40              40.90
 2017      PE/Debt    376       10,355.10              54.70
 2017         Seed    311           74.20              45.30
 2018      PE/Debt    163        4,266.60              52.80
 2018         Seed    126          222.50              40.80
 2018     Series A      8   

In [26]:
query_11 = """-- Query 11: Funding Stage Evolution By Year
-- Business question: Did ecosystem shift from seed-heavy to growth-stage capital?
-- Concepts: GROUP BY multiple columns, window function percentage within year

WITH stage_year AS (
    SELECT
        CAST(year AS INTEGER)  AS year,
        stage_clean,
        COUNT(*)               AS deals,
        ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
    FROM funding_rounds
    WHERE year >= 2014
      AND year <= 2020
      AND stage_clean IS NOT NULL
      AND stage_clean != 'Other'
    GROUP BY year, stage_clean
)
SELECT
    year,
    stage_clean,
    deals,
    funding_mn_usd,
    ROUND(
        deals * 100.0 /
        SUM(deals) OVER (PARTITION BY year)
    , 1)                       AS pct_of_year_deals
FROM stage_year
ORDER BY year, deals DESC;
"""

with open('../sql/11_stage_evolution.sql', 'w') as f:
    f.write(query_11)

print("✓ Saved: sql/11_stage_evolution.sql")

✓ Saved: sql/11_stage_evolution.sql


---

## Query 12 — City Funding Concentration By Year

**Business question:** Is Bengaluru's dominance growing or declining over time?

**SQL concepts used:** `PARTITION BY` two columns, `RANK()`, subquery filtering

In [27]:
q12 = run_query("""
    WITH city_year AS (
        SELECT
            CAST(year AS INTEGER)         AS year,
            city_clean,
            COUNT(*)                      AS deals,
            ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
        FROM funding_rounds
        WHERE year >= 2015
          AND year <= 2019
          AND city_clean IS NOT NULL
          AND amount_usd IS NOT NULL
        GROUP BY year, city_clean
    ),
    ranked AS (
        SELECT
            year,
            city_clean,
            deals,
            funding_mn_usd,
            ROUND(
                funding_mn_usd * 100.0 /
                SUM(funding_mn_usd) OVER (PARTITION BY year)
            , 1)                          AS pct_of_year_funding,
            RANK() OVER (
                PARTITION BY year
                ORDER BY funding_mn_usd DESC
            )                             AS city_rank
        FROM city_year
    )
    SELECT *
    FROM ranked
    WHERE city_rank <= 5
    ORDER BY year, city_rank
""", "Top 5 cities by funding per year")

print(q12.to_string(index=False))

Query: Top 5 cities by funding per year
Returned 25 rows

 year city_clean  deals  funding_mn_usd  pct_of_year_funding  city_rank
 2015  Bengaluru    137        2,951.20                40.30          1
 2015      Delhi     80        1,747.50                23.90          2
 2015    Gurgaon     66          982.40                13.40          3
 2015     Mumbai    115          981.70                13.40          4
 2015    Chennai     15          241.90                 3.30          5
 2016  Bengaluru    165        1,023.40                26.70          1
 2016     Mumbai    117          921.10                24.10          2
 2016      Delhi    101          790.20                20.60          3
 2016    Gurgaon     65          637.70                16.70          4
 2016    Chennai     22          116.80                 3.10          5
 2017  Bengaluru    161        7,416.60                71.10          1
 2017     Mumbai     99        1,161.50                11.10          2
 2017 

In [28]:
query_12 = """-- Query 12: City Funding Concentration By Year
-- Business question: Is Bengaluru's dominance growing or declining?
-- Concepts: RANK() with PARTITION BY year, percentage within year window function

WITH city_year AS (
    SELECT
        CAST(year AS INTEGER)         AS year,
        city_clean,
        COUNT(*)                      AS deals,
        ROUND(SUM(amount_usd)/1e6, 1) AS funding_mn_usd
    FROM funding_rounds
    WHERE year >= 2015
      AND year <= 2019
      AND city_clean IS NOT NULL
      AND amount_usd IS NOT NULL
    GROUP BY year, city_clean
),
ranked AS (
    SELECT
        year,
        city_clean,
        deals,
        funding_mn_usd,
        ROUND(
            funding_mn_usd * 100.0 /
            SUM(funding_mn_usd) OVER (PARTITION BY year)
        , 1)                          AS pct_of_year_funding,
        RANK() OVER (
            PARTITION BY year
            ORDER BY funding_mn_usd DESC
        )                             AS city_rank
    FROM city_year
)
SELECT *
FROM ranked
WHERE city_rank <= 5
ORDER BY year, city_rank;
"""

with open('../sql/12_city_concentration.sql', 'w') as f:
    f.write(query_12)

print("✓ Saved: sql/12_city_concentration.sql")

✓ Saved: sql/12_city_concentration.sql


---

## Query 13 — Fintech Resilience Across Downturns

**Business question:** Did Fintech hold up better than other sectors during the 2016 and 2018 downturns?

**SQL concepts used:** `CASE` for downturn flagging, `FILTER` clause, comparative aggregation

This directly tests hypothesis: *"Fintech is the most resilient sector."*

In [29]:
q13 = run_query("""
    WITH sector_era AS (
        SELECT
            sector,
            CASE
                WHEN CAST(year AS INTEGER) IN (2015) THEN 'Boom'
                WHEN CAST(year AS INTEGER) IN (2016) THEN 'Downturn 1'
                WHEN CAST(year AS INTEGER) IN (2017) THEN 'Recovery'
                WHEN CAST(year AS INTEGER) IN (2018) THEN 'Downturn 2'
                WHEN CAST(year AS INTEGER) IN (2019) THEN 'Late Cycle'
            END                            AS market_phase,
            amount_usd
        FROM funding_rounds
        WHERE year BETWEEN 2015 AND 2019
          AND sector IS NOT NULL
          AND amount_usd IS NOT NULL
    ),
    phase_totals AS (
        SELECT
            sector,
            market_phase,
            COUNT(*)                       AS deals,
            ROUND(SUM(amount_usd)/1e6, 1)  AS funding_mn_usd
        FROM sector_era
        WHERE market_phase IS NOT NULL
        GROUP BY sector, market_phase
    )
    SELECT
        sector,
        MAX(CASE WHEN market_phase = 'Boom'       THEN funding_mn_usd END) AS boom_2015,
        MAX(CASE WHEN market_phase = 'Downturn 1' THEN funding_mn_usd END) AS downturn1_2016,
        MAX(CASE WHEN market_phase = 'Recovery'   THEN funding_mn_usd END) AS recovery_2017,
        MAX(CASE WHEN market_phase = 'Downturn 2' THEN funding_mn_usd END) AS downturn2_2018,
        MAX(CASE WHEN market_phase = 'Late Cycle' THEN funding_mn_usd END) AS late_2019
    FROM phase_totals
    WHERE sector IN (
        'Fintech', 'Edtech', 'Healthtech',
        'E-commerce', 'Foodtech', 'SaaS/B2B',
        'Mobility', 'Logistics'
    )
    GROUP BY sector
    ORDER BY late_2019 DESC
""", "Sector funding across market phases")

print(q13.to_string(index=False))

Query: Sector funding across market phases
Returned 8 rows

    sector  boom_2015  downturn1_2016  recovery_2017  downturn2_2018  late_2019
  Mobility     718.30             NaN            NaN          304.00   4,540.90
   Fintech     137.60             NaN            NaN          144.10   1,221.20
E-commerce   3,315.10          975.00       5,937.80          155.50   1,154.90
  SaaS/B2B     118.30             NaN            NaN          256.50     977.00
Healthtech     204.40           52.40         116.00          340.00     503.00
    Edtech      82.50           82.60           8.20           44.70     359.40
  Foodtech     605.40           23.70          18.70          315.40      34.70
 Logistics     293.00           38.10         184.70           37.70        NaN


In [30]:
query_13 = """-- Query 13: Sector Resilience Across Market Phases
-- Business question: Which sectors held up during the 2016 and 2018 downturns?
-- Concepts: CASE for phase labeling, pivot using MAX(CASE WHEN...) pattern

WITH sector_era AS (
    SELECT
        sector,
        CASE
            WHEN CAST(year AS INTEGER) = 2015 THEN 'Boom'
            WHEN CAST(year AS INTEGER) = 2016 THEN 'Downturn 1'
            WHEN CAST(year AS INTEGER) = 2017 THEN 'Recovery'
            WHEN CAST(year AS INTEGER) = 2018 THEN 'Downturn 2'
            WHEN CAST(year AS INTEGER) = 2019 THEN 'Late Cycle'
        END                            AS market_phase,
        amount_usd
    FROM funding_rounds
    WHERE year BETWEEN 2015 AND 2019
      AND sector IS NOT NULL
      AND amount_usd IS NOT NULL
),
phase_totals AS (
    SELECT
        sector,
        market_phase,
        COUNT(*)                       AS deals,
        ROUND(SUM(amount_usd)/1e6, 1)  AS funding_mn_usd
    FROM sector_era
    WHERE market_phase IS NOT NULL
    GROUP BY sector, market_phase
)
SELECT
    sector,
    MAX(CASE WHEN market_phase = 'Boom'       THEN funding_mn_usd END) AS boom_2015,
    MAX(CASE WHEN market_phase = 'Downturn 1' THEN funding_mn_usd END) AS downturn1_2016,
    MAX(CASE WHEN market_phase = 'Recovery'   THEN funding_mn_usd END) AS recovery_2017,
    MAX(CASE WHEN market_phase = 'Downturn 2' THEN funding_mn_usd END) AS downturn2_2018,
    MAX(CASE WHEN market_phase = 'Late Cycle' THEN funding_mn_usd END) AS late_2019
FROM phase_totals
WHERE sector IN (
    'Fintech', 'Edtech', 'Healthtech',
    'E-commerce', 'Foodtech', 'SaaS/B2B',
    'Mobility', 'Logistics'
)
GROUP BY sector
ORDER BY late_2019 DESC;
"""

with open('../sql/13_sector_resilience.sql', 'w') as f:
    f.write(query_13)

print("✓ Saved: sql/13_sector_resilience.sql")

✓ Saved: sql/13_sector_resilience.sql


---

## Query 14 — Cohort Analysis: Companies By First Funding Year

**Business question:** Of companies that first raised in the 2015 boom — how many raised again in subsequent years?

**SQL concepts used:** CTE, `MIN()` to find first funding year, cohort grouping

### What is Cohort Analysis?
Group entities by when they first appeared, then track their behavior over time.
Here: companies grouped by first funding year — did they raise follow-on rounds?


In [31]:
q14 = run_query("""
    WITH first_funding AS (
        SELECT
            startup_name_clean,
            CAST(MIN(year) AS INTEGER)    AS cohort_year
        FROM funding_rounds
        WHERE year >= 2013
          AND year <= 2019
          AND startup_name_clean IS NOT NULL
        GROUP BY startup_name_clean
    ),
    cohort_activity AS (
        SELECT
            f.cohort_year,
            COUNT(DISTINCT f.startup_name_clean)  AS cohort_size,
            COUNT(DISTINCT CASE
                WHEN CAST(r.year AS INTEGER) > f.cohort_year
                THEN f.startup_name_clean
            END)                                  AS raised_followon
        FROM first_funding f
        LEFT JOIN funding_rounds r
            ON f.startup_name_clean = r.startup_name_clean
        GROUP BY f.cohort_year
    )
    SELECT
        cohort_year,
        cohort_size,
        raised_followon,
        ROUND(
            raised_followon * 100.0 / cohort_size
        , 1)                                      AS followon_rate_pct
    FROM cohort_activity
    ORDER BY cohort_year
""", "Cohort analysis — follow-on funding rate by first funding year")

print(q14.to_string(index=False))

Query: Cohort analysis — follow-on funding rate by first funding year
Returned 7 rows

 cohort_year  cohort_size  raised_followon  followon_rate_pct
        2013          158               24              15.20
        2014          161               35              21.70
        2015          804              210              26.10
        2016          767              102              13.30
        2017          455               26               5.70
        2018          186                5               2.70
        2019           64                1               1.60


In [32]:
query_14 = """-- Query 14: Cohort Analysis — Follow-on Funding Rate
-- Business question: Of companies that first raised in boom years, how many raised again?
-- Concepts: MIN() for cohort definition, LEFT JOIN for activity tracking, cohort retention rate

WITH first_funding AS (
    SELECT
        startup_name_clean,
        CAST(MIN(year) AS INTEGER)    AS cohort_year
    FROM funding_rounds
    WHERE year >= 2013
      AND year <= 2019
      AND startup_name_clean IS NOT NULL
    GROUP BY startup_name_clean
),
cohort_activity AS (
    SELECT
        f.cohort_year,
        COUNT(DISTINCT f.startup_name_clean)  AS cohort_size,
        COUNT(DISTINCT CASE
            WHEN CAST(r.year AS INTEGER) > f.cohort_year
            THEN f.startup_name_clean
        END)                                  AS raised_followon
    FROM first_funding f
    LEFT JOIN funding_rounds r
        ON f.startup_name_clean = r.startup_name_clean
    GROUP BY f.cohort_year
)
SELECT
    cohort_year,
    cohort_size,
    raised_followon,
    ROUND(
        raised_followon * 100.0 / cohort_size
    , 1)                                      AS followon_rate_pct
FROM cohort_activity
ORDER BY cohort_year;
"""

with open('../sql/14_cohort_analysis.sql', 'w') as f:
    f.write(query_14)

print("✓ Saved: sql/14_cohort_analysis.sql")

✓ Saved: sql/14_cohort_analysis.sql


---

## Query 15 — Quarterly Funding Velocity

**Business question:** Which quarters saw the sharpest acceleration or deceleration in deal activity?

**SQL concepts used:** CTE, LAG() on quarter-level data, quarter-over-quarter change

In [33]:
q15 = run_query("""
    WITH quarterly AS (
        SELECT
            CAST(year AS INTEGER)             AS year,
            CAST(quarter AS INTEGER)          AS quarter,
            year || '-Q' || CAST(quarter AS INTEGER) AS year_quarter,
            COUNT(*)                          AS deals,
            ROUND(SUM(amount_usd)/1e6, 1)     AS funding_mn_usd
        FROM funding_rounds
        WHERE year >= 2015
          AND year <= 2019
          AND quarter IS NOT NULL
          AND amount_usd IS NOT NULL
        GROUP BY year, quarter
    )
    SELECT
        year_quarter,
        deals,
        funding_mn_usd,
        LAG(deals, 1) OVER (
            ORDER BY year, quarter
        )                                     AS prev_quarter_deals,
        deals - LAG(deals, 1) OVER (
            ORDER BY year, quarter
        )                                     AS deal_change,
        ROUND(
            (deals - LAG(deals, 1) OVER (
                ORDER BY year, quarter
            )) * 100.0 /
            LAG(deals, 1) OVER (
                ORDER BY year, quarter
            )
        , 1)                                  AS qoq_growth_pct
    FROM quarterly
    ORDER BY year, quarter
""", "Quarterly funding velocity")

print(q15.to_string(index=False))

Query: Quarterly funding velocity
Returned 20 rows

year_quarter  deals  funding_mn_usd  prev_quarter_deals  deal_change  qoq_growth_pct
   2015.0-Q1    127        1,226.90                 NaN          NaN             NaN
   2015.0-Q2    152        1,841.70              127.00        25.00           19.70
   2015.0-Q3    188        4,068.50              152.00        36.00           23.70
   2015.0-Q4    175        1,460.10              188.00       -13.00           -6.90
   2016.0-Q1    166        1,341.00              175.00        -9.00           -5.10
   2016.0-Q2    144          793.40              166.00       -22.00          -13.30
   2016.0-Q3    140        1,009.00              144.00        -4.00           -2.80
   2016.0-Q4    136          684.70              140.00        -4.00           -2.90
   2017.0-Q1    124        2,812.50              136.00       -12.00           -8.80
   2017.0-Q2    134        2,863.90              124.00        10.00            8.10
   2017.0-Q3 

In [34]:
query_15 = """-- Query 15: Quarterly Funding Velocity
-- Business question: Which quarters saw sharpest acceleration or deceleration?
-- Concepts: Quarter-level CTE, LAG() for QoQ change, string concatenation for labels

WITH quarterly AS (
    SELECT
        CAST(year AS INTEGER)             AS year,
        CAST(quarter AS INTEGER)          AS quarter,
        year || '-Q' || CAST(quarter AS INTEGER) AS year_quarter,
        COUNT(*)                          AS deals,
        ROUND(SUM(amount_usd)/1e6, 1)     AS funding_mn_usd
    FROM funding_rounds
    WHERE year >= 2015
      AND year <= 2019
      AND quarter IS NOT NULL
      AND amount_usd IS NOT NULL
    GROUP BY year, quarter
)
SELECT
    year_quarter,
    deals,
    funding_mn_usd,
    LAG(deals, 1) OVER (
        ORDER BY year, quarter
    )                                     AS prev_quarter_deals,
    deals - LAG(deals, 1) OVER (
        ORDER BY year, quarter
    )                                     AS deal_change,
    ROUND(
        (deals - LAG(deals, 1) OVER (
            ORDER BY year, quarter
        )) * 100.0 /
        LAG(deals, 1) OVER (
            ORDER BY year, quarter
        )
    , 1)                                  AS qoq_growth_pct
FROM quarterly
ORDER BY year, quarter;
"""

with open('../sql/15_quarterly_velocity.sql', 'w') as f:
    f.write(query_15)

print("✓ Saved: sql/15_quarterly_velocity.sql")
print("\n✅ All 15 queries complete.")
print("\nSQL files saved:")
import os
sql_files = sorted(os.listdir('../sql/'))
for f in sql_files:
    if f.endswith('.sql'):
        print(f"  {f}")

✓ Saved: sql/15_quarterly_velocity.sql

✅ All 15 queries complete.

SQL files saved:
  01_yearly_trends.sql
  02_sector_evolution.sql
  03_top_cities.sql
  04_yoy_growth.sql
  05_sector_rankings.sql
  06_top_company_per_sector.sql
  07_deal_size_buckets.sql
  08_cumulative_funding.sql
  09_investor_activity.sql
  10_median_vs_mean.sql
  11_stage_evolution.sql
  12_city_concentration.sql
  13_sector_resilience.sql
  14_cohort_analysis.sql
  15_quarterly_velocity.sql
